In [1]:
import pandas
import numpy as np

In [3]:
# example
cons_model = "split_cds_regulatory"
mut_model = "roulette"
chrom = 1
prefix = "../local_models/tables_10kb/equilibrium_granular_Ne"

fname = f"{prefix}/{cons_model}/{mut_model}/{cons_model}_{mut_model}_chr{chrom}_log.txt"
with open(fname, "r") as fin:
    header = fin.readline()
fields = {x.strip(): float(y) for x, y in 
          [a.split(": ") for a in header.lstrip("#").rstrip("\n").split(",")]}
print(fields)

{'Ne_opt': 25910.15625, 'f_opt': 29707507814.187256, 'n_iters': 10.0, 'n_calls': 20.0, 'flag': 1.0}


In [5]:
cons_models = [
    "merged_cds",
    "split_cds",
    "merged_cds_regulatory",
    "split_cds_regulatory",
    "merged_cds_phastcons",
    "split_cds_phastcons"]
mut_models = [
    "roulette",
    "gnomad",
    "carlson"]
chroms = list(range(1, 23))
prefix = "../local_models/tables_10kb/equilibrium_granular_Ne"

cols = [
    "cons_model",
    "mut_model",
    "chrom",
    "Ne",
    "ll"]
data = []

for cons_model in cons_models:
    for mut_model in mut_models:
        for chrom in chroms:
            # Load the header of the log file
            fname = f"{prefix}/{cons_model}/{mut_model}/{cons_model}_{mut_model}_chr{chrom}_log.txt"
            with open(fname, "r") as fin:
                header = fin.readline()
            fields = {x.strip(): float(y) for x, y in 
                      [a.split(": ") for a in header.lstrip("#").rstrip("\n").split(",")]}
            Ne = fields["Ne_opt"]
            ll = -fields["f_opt"]
            row = [
                cons_model,
                mut_model,
                chrom,
                Ne,
                ll]
            data.append(row)
df = pandas.DataFrame(data, columns=cols)
df.to_csv("../models/granular_Ne_tbl.csv", index=False)

In [7]:
# Compute a genome-wide Ne for each model by taking a weighted average of granular Ne
cons_models = [
    "merged_cds",
    "split_cds",
    "merged_cds_regulatory",
    "split_cds_regulatory",
    "merged_cds_phastcons",
    "split_cds_phastcons"]
mut_models = [
    "roulette",
    "gnomad",
    "carlson"]
chroms = list(range(1, 23))

Ne_df = pandas.read_csv("../models/granular_Ne_tbl.csv")
num_site_df = pandas.read_csv("../data/site_count_tbl.csv")

cols = [
    "cons_model",
    "mut_model",
    "avg_Ne"]
data = []

for cons_model in cons_models:
    for mut_model in mut_models:
        numer = 0.0
        denom = 0.0
        for chrom in chroms:
            Ne = next(iter(Ne_df[
                (Ne_df["cons_model"] == cons_model)
                & (Ne_df["mut_model"] == mut_model)
                & (Ne_df["chrom"] == chrom)]["Ne"]))
            n_sites = next(iter(num_site_df[
                (num_site_df["mut_model"] == mut_model)
                & (num_site_df["chrom"] == chrom)]["num_sites"]))
            numer += Ne * n_sites
            denom += n_sites
        avg_Ne = numer / denom
        row = [
            cons_model,
            mut_model,
            avg_Ne]
        data.append(row)
df = pandas.DataFrame(data, columns=cols)
df.to_csv("../models/avg_Ne_tbl.csv", index=False)
print(df.to_string(index=False))

           cons_model mut_model       avg_Ne
           merged_cds  roulette 23767.056074
           merged_cds    gnomad 24889.082643
           merged_cds   carlson 18717.457782
            split_cds  roulette 24454.541705
            split_cds    gnomad 25649.535683
            split_cds   carlson 19310.022313
merged_cds_regulatory  roulette 25903.136128
merged_cds_regulatory    gnomad 27310.491914
merged_cds_regulatory   carlson 20749.483281
 split_cds_regulatory  roulette 26657.917091
 split_cds_regulatory    gnomad 28155.433521
 split_cds_regulatory   carlson 21417.578591
 merged_cds_phastcons  roulette 33021.729369
 merged_cds_phastcons    gnomad 35095.518996
 merged_cds_phastcons   carlson 27641.026274
  split_cds_phastcons  roulette 34029.227037
  split_cds_phastcons    gnomad 36223.463533
  split_cds_phastcons   carlson 28566.591032


In [2]:
# Take autosome-wide likelihoods
cons_models = [
    "merged_cds_regulatory",
    "split_cds_regulatory",
    "merged_cds_phastcons",
    "split_cds_phastcons"]
mut_models = [
    "carlson",
    "gnomad",
    "roulette"]
chroms = list(range(1, 23))

Ne_df = pandas.read_csv("../models/granular_Ne_tbl.csv")
num_site_df = pandas.read_csv("../data/site_count_tbl.csv")

cols = [
    "cons_model",
    "mut_model",
    "num_sites",
    "ll"]
data = []

for cons_model in cons_models:
    for mut_model in mut_models:
        tot_ll = 0.0
        tot_n_sites = 0
        for chrom in chroms:
            ll = next(iter(Ne_df[
                (Ne_df["cons_model"] == cons_model)
                & (Ne_df["mut_model"] == mut_model)
                & (Ne_df["chrom"] == chrom)]["ll"]))
            n_sites = next(iter(num_site_df[
                (num_site_df["mut_model"] == mut_model)
                & (num_site_df["chrom"] == chrom)]["num_sites"]))
            tot_ll += ll
            tot_n_sites += n_sites
        row = [
            cons_model,
            mut_model,
            tot_n_sites,
            tot_ll]
        data.append(row)
df = pandas.DataFrame(data, columns=cols)
df.to_csv("../models/granular_autosomal_ll_tbl.csv", index=False)
print(df.to_string(index=False))

           cons_model mut_model  num_sites            ll
merged_cds_regulatory   carlson 1968399089 -3.396618e+11
merged_cds_regulatory    gnomad 2164673425 -3.840134e+11
merged_cds_regulatory  roulette 2164673425 -3.831660e+11
 split_cds_regulatory   carlson 1968399089 -3.396689e+11
 split_cds_regulatory    gnomad 2164673425 -3.840302e+11
 split_cds_regulatory  roulette 2164673425 -3.831732e+11
 merged_cds_phastcons   carlson 1968399089 -3.391252e+11
 merged_cds_phastcons    gnomad 2164673425 -3.831838e+11
 merged_cds_phastcons  roulette 2164673425 -3.826512e+11
  split_cds_phastcons   carlson 1968399089 -3.391061e+11
  split_cds_phastcons    gnomad 2164673425 -3.831728e+11
  split_cds_phastcons  roulette 2164673425 -3.826344e+11


In [8]:
df = df[df["mut_model"] != "carlson"]
df["deltaLL"] = np.max(df["ll"]) - np.array(df["ll"]) 
print(df.to_string(index=False))

           cons_model mut_model  num_sites            ll      deltaLL
merged_cds_regulatory    gnomad 2164673425 -3.840134e+11 1.378994e+09
merged_cds_regulatory  roulette 2164673425 -3.831660e+11 5.316324e+08
 split_cds_regulatory    gnomad 2164673425 -3.840302e+11 1.395750e+09
 split_cds_regulatory  roulette 2164673425 -3.831732e+11 5.388178e+08
 merged_cds_phastcons    gnomad 2164673425 -3.831838e+11 5.494125e+08
 merged_cds_phastcons  roulette 2164673425 -3.826512e+11 1.682158e+07
  split_cds_phastcons    gnomad 2164673425 -3.831728e+11 5.384178e+08
  split_cds_phastcons  roulette 2164673425 -3.826344e+11 0.000000e+00
